In [15]:
"""
hurst_analysis.py
-----------------
Analisi dell'esponente di Hurst su dati OHLCV.

Tutti i parametri interni (finestre, stride, lag VR) sono espressi in ore
e convertiti automaticamente in candele in base al timeframe della serie.
Questo garantisce che la stessa analisi su 1h, 4h o 1d produca finestre
temporalmente coerenti e output comparabili.

Produce:
  - H rolling (DFA) sulla serie dei prezzi
  - Etichetta stabile del ticker (mediana storica di H)
  - Half-life OU se mean reverting, con lookback window suggerita
  - Variance Ratio profile per identificare la scala temporale dominante

Input atteso:
  pd.DataFrame con colonne [open_time, open, high, low, close, volume]
  open_time deve essere datetime o ordinabile cronologicamente

Utilizzo:
    from hurst_analysis import analyze_ticker
    result = analyze_ticker(df, timeframe='1h')
    result = analyze_ticker(df, timeframe='4h')
    result = analyze_ticker(df, timeframe='1d')
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm


# ---------------------------------------------------------------------------
# 0. TIMEFRAME UTILITIES
# ---------------------------------------------------------------------------

# Mapping timeframe → ore per candela
TIMEFRAME_HOURS: dict[str, float] = {
    '1m':   1 / 60,
    '5m':   5 / 60,
    '15m':  15 / 60,
    '30m':  0.5,
    '1h':   1.0,
    '2h':   2.0,
    '4h':   4.0,
    '6h':   6.0,
    '8h':   8.0,
    '12h':  12.0,
    '1d':   24.0,
    '3d':   72.0,
    '1w':   168.0,
}


def tf_hours(timeframe: str) -> float:
    """Ritorna le ore per candela del timeframe dato."""
    tf = timeframe.lower().strip()
    if tf not in TIMEFRAME_HOURS:
        raise ValueError(
            f"Timeframe '{timeframe}' non riconosciuto. "
            f"Valori validi: {list(TIMEFRAME_HOURS.keys())}"
        )
    return TIMEFRAME_HOURS[tf]


def hours_to_candles(hours: float, tf: str) -> int:
    """Converte un numero di ore in candele per il timeframe dato."""
    return max(1, int(np.round(hours / tf_hours(tf))))


def candles_to_hours(candles: int, tf: str) -> float:
    """Converte un numero di candele in ore per il timeframe dato."""
    return candles * tf_hours(tf)


def format_duration(hours: float) -> str:
    """Formatta una durata in ore in una stringa leggibile."""
    if hours < 1:
        return f"{hours * 60:.0f}m"
    if hours < 24:
        return f"{hours:.1f}h"
    if hours < 168:
        return f"{hours / 24:.1f}d"
    return f"{hours / 168:.1f}w"


# ---------------------------------------------------------------------------
# 1. PARAMETRI DEFAULT DIPENDENTI DAL TIMEFRAME
# ---------------------------------------------------------------------------

def default_params(timeframe: str) -> dict:
    """
    Ritorna i parametri operativi espressi in ore — indipendenti dal timeframe.
    La conversione in candele avviene in analyze_ticker.

    Il parametro `hurst_min_window_candles` non può essere espresso in ore
    perché è un floor interno al DFA che dipende dal numero di punti nella
    finestra di stima. Viene calcolato automaticamente in analyze_ticker come:

        min_window_c = max(HURST_MIN_WINDOW_FLOOR, window_c // HURST_MIN_WINDOW_RATIO)

    dove:
        HURST_MIN_WINDOW_FLOOR = 4   → floor assoluto: almeno 4 punti per segmento
        HURST_MIN_WINDOW_RATIO = 10  → min_window ≈ 10% della finestra rolling

    Esempi per timeframe diversi:
        1h  → window=504c  →  min_window = max(4, 50) = 50
        4h  → window=126c  →  min_window = max(4, 12) = 12
        1d  → window= 90c  →  min_window = max(4,  9) =  9

    Il vincolo del DFA è max_window (= window_c//4) >= min_window * 2.
    Con finestra 21g su 1d → window_c=21 → max_window=5 → min_window=2 ma 5 < 4
    Per questo motivo su timeframe >= 1d la finestra viene allargata a 90 giorni.
    """
    h = tf_hours(timeframe)

    # Per timeframe >= 1d la finestra di 21 giorni genera troppo poche candele
    # per il DFA — allarga automaticamente a 90 giorni
    window_hours = 90 * 24 if h >= 24 else 21 * 24

    return {
        'hurst_window_hours':       window_hours,
        'hurst_stride_hours':       1 * 24,   # 1 giorno
        'hurst_min_window_candles': None,      # None = calcolato automaticamente
        'vr_lag_hours': [4, 8, 16, 24, 48, 96, 168],
    }


# Costanti per il calcolo automatico di min_window nel DFA
HURST_MIN_WINDOW_FLOOR = 4    # floor assoluto in candele
HURST_MIN_WINDOW_RATIO = 10   # min_window ≈ window_c / RATIO


# ---------------------------------------------------------------------------
# 2. HURST EXPONENT — Detrended Fluctuation Analysis
# ---------------------------------------------------------------------------

def hurst_dfa(series: np.ndarray, min_window: int = 10, max_window: int = None) -> float:
    """
    Stima l'esponente di Hurst tramite DFA (Detrended Fluctuation Analysis).

    Parameters
    ----------
    series     : array di prezzi (non rendimenti)
    min_window : finestra minima interna in candele
    max_window : finestra massima interna in candele (default: len/4)

    Returns
    -------
    H : float
        < 0.5  →  mean reverting
        = 0.5  →  random walk
        > 0.5  →  momentum / trending
    """
    log_ret = np.diff(np.log(series))
    N       = len(log_ret)

    if max_window is None:
        max_window = N // 4

    if max_window < min_window * 2:
        raise ValueError(
            f"Serie troppo corta per DFA: {N} rendimenti disponibili, "
            f"max_window={max_window} < min_window*2={min_window*2}"
        )

    windows = np.unique(np.logspace(
        np.log10(min_window),
        np.log10(max_window),
        num=20,
        dtype=int
    ))

    fluct = []
    for w in windows:
        segments = [log_ret[i:i+w] for i in range(0, N - w + 1, w)]
        rms_segs = []
        for seg in segments:
            if len(seg) < 4:
                continue
            x     = np.arange(len(seg), dtype=float)
            coeff = np.polyfit(x, seg, 1)
            det   = seg - np.polyval(coeff, x)
            rms_segs.append(np.sqrt(np.mean(det ** 2)))
        fluct.append(np.mean(rms_segs) if rms_segs else np.nan)

    valid = [
        (np.log(w), np.log(f))
        for w, f in zip(windows, fluct)
        if not np.isnan(f) and f > 0
    ]

    if len(valid) < 3:
        return np.nan

    log_w  = np.array([v[0] for v in valid])
    log_f  = np.array([v[1] for v in valid])
    coeffs = np.polyfit(log_w, log_f, 1)
    return float(coeffs[0])


# ---------------------------------------------------------------------------
# 3. HALF-LIFE — Ornstein-Uhlenbeck discreto
# ---------------------------------------------------------------------------

def ou_halflife(series: np.ndarray) -> float | None:
    """
    Stima il half-life di mean reversion tramite fit OU discreto.
    Ritorna il valore in candele, oppure None se il processo non è mean reverting.

    dP_t = kappa * (mu - P_(t-1)) + sigma * dW_t
    half_life = -log(2) / log(1 + kappa)
    """
    log_p = np.log(series)
    delta = np.diff(log_p)
    lag   = log_p[:-1]

    X     = sm.add_constant(lag)
    model = sm.OLS(delta, X).fit()
    kappa = float(model.params[1])

    if kappa >= 0:
        return None

    hl = -np.log(2) / np.log(1 + kappa)

    if hl <= 0 or hl > len(series):
        return None

    return float(hl)


# ---------------------------------------------------------------------------
# 4. VARIANCE RATIO PROFILE
# ---------------------------------------------------------------------------

def variance_ratio_profile(series: np.ndarray, lags_candles: list[int]) -> dict:
    """
    Calcola il Variance Ratio per ogni lag (espresso in candele).

    VR > 1  →  momentum
    VR = 1  →  random walk
    VR < 1  →  mean reversion

    Parameters
    ----------
    series       : array di prezzi
    lags_candles : lista di lag in candele

    Returns
    -------
    dict {lag_candles: vr_value}
    """
    log_ret = np.diff(np.log(series))
    var_1   = np.var(log_ret, ddof=1)

    if var_1 == 0:
        return {lag: np.nan for lag in lags_candles}

    results = {}
    for lag in lags_candles:
        if lag >= len(log_ret):
            results[lag] = np.nan
            continue
        ret_lag      = np.array([log_ret[i:i+lag].sum() for i in range(len(log_ret) - lag + 1)])
        var_lag      = np.var(ret_lag, ddof=1)
        results[lag] = float(var_lag / (lag * var_1))

    return results


# ---------------------------------------------------------------------------
# 5. ROLLING HURST
# ---------------------------------------------------------------------------

def rolling_hurst(
    prices:             pd.Series,
    window_candles:     int,
    stride_candles:     int,
    min_window_candles: int = 10,
) -> pd.Series:
    """
    Calcola H su finestre rolling. Tutti i parametri sono in candele —
    la conversione da ore a candele avviene in analyze_ticker.

    Parameters
    ----------
    prices             : pd.Series prezzi, indice = open_time
    window_candles     : ampiezza della finestra rolling in candele
    stride_candles     : passo tra una stima e la successiva in candele
    min_window_candles : floor interno al DFA

    Returns
    -------
    pd.Series con H stimato, indice = timestamp fine finestra
    """
    prices = prices.sort_index()
    values = prices.values
    times  = prices.index
    N      = len(values)

    results = {}
    for i in range(window_candles, N + 1, stride_candles):
        segment = values[i - window_candles:i]
        ts      = times[i - 1]
        try:
            H = hurst_dfa(segment, min_window=min_window_candles)
        except Exception:
            H = np.nan
        results[ts] = H

    return pd.Series(results, name='H')


# ---------------------------------------------------------------------------
# 6. ETICHETTATURA TICKER
# ---------------------------------------------------------------------------

def label_ticker(H_series: pd.Series) -> dict:
    """
    Produce l'etichetta stabile del ticker dalla serie rolling di H.
    Applica smoothing con mediana mobile a 5 passi prima del calcolo.

    Returns
    -------
    dict con label, H_median, H_std, H_current, H_ci, regime_drift
    """
    H_clean = H_series.dropna()

    if len(H_clean) < 5:
        return {
            'label':        'INSUFFICIENT_DATA',
            'H_median':     np.nan,
            'H_std':        np.nan,
            'H_current':    np.nan,
            'H_ci':         (np.nan, np.nan),
            'regime_drift': False,
        }

    H_smooth  = H_clean.rolling(5, min_periods=1).median()
    H_median  = float(H_smooth.median())
    H_std     = float(H_smooth.std())
    H_current = float(H_clean.iloc[-1])
    H_ci_low  = H_median - H_std
    H_ci_hi   = H_median + H_std

    if H_ci_low < 0.5 < H_ci_hi:
        label = 'MIXED'
    elif H_median < 0.40:
        label = 'MEAN_REVERTING_STRONG'
    elif H_median < 0.48:
        label = 'MEAN_REVERTING'
    elif H_median <= 0.52:
        label = 'RANDOM'
    elif H_median <= 0.60:
        label = 'MOMENTUM'
    else:
        label = 'MOMENTUM_STRONG'

    regime_drift = abs(H_current - H_median) > 1.5 * H_std

    return {
        'label':        label,
        'H_median':     round(H_median, 4),
        'H_std':        round(H_std, 4),
        'H_current':    round(H_current, 4),
        'H_ci':         (round(H_ci_low, 4), round(H_ci_hi, 4)),
        'regime_drift': regime_drift,
    }


# ---------------------------------------------------------------------------
# 7. PIPELINE COMPLETA
# ---------------------------------------------------------------------------

def analyze_ticker(
    df:        pd.DataFrame,
    timeframe: str  = '1h',
    price_col: str  = 'close',
    params:    dict = None,
) -> dict:
    """
    Pipeline completa timeframe-aware.

    Tutti i parametri interni (finestre, stride, lag VR) sono espressi in ore
    e convertiti in candele in base al timeframe della serie. Questo garantisce
    che la stessa analisi su dati 1h, 4h o 1d copra gli stessi orizzonti temporali.

    Parameters
    ----------
    df        : DataFrame con colonne OHLCV e open_time
    timeframe : stringa timeframe, es. '1h', '4h', '1d'
    price_col : colonna prezzi (default 'close')
    params    : override dei parametri default (in ore) — vedi default_params()

    Returns
    -------
    dict con tutti i risultati + metadati sul timeframe usato
    """
    # --- Setup timeframe ---
    h_per_candle = tf_hours(timeframe)
    p            = default_params(timeframe)
    if params:
        p.update(params)

    # Conversione ore → candele per tutti i parametri
    window_c  = hours_to_candles(p['hurst_window_hours'], timeframe)
    stride_c  = hours_to_candles(p['hurst_stride_hours'], timeframe)

    # min_window: calcolato automaticamente se non specificato esplicitamente
    if p['hurst_min_window_candles'] is None:
        min_win_c = max(HURST_MIN_WINDOW_FLOOR, window_c // HURST_MIN_WINDOW_RATIO)
    else:
        min_win_c = p['hurst_min_window_candles']

    vr_lags_c    = [
        hours_to_candles(h, timeframe)
        for h in p['vr_lag_hours']
        if hours_to_candles(h, timeframe) >= 2
    ]
    # Deduplication mantenendo l'ordine
    seen = set()
    vr_lags_c = [x for x in vr_lags_c if not (x in seen or seen.add(x))]

    # --- Preparazione dati ---
    df     = df.sort_values('open_time').reset_index(drop=True)
    prices = df.set_index('open_time')[price_col].astype(float)

    min_required = window_c + 50
    if len(prices) < min_required:
        raise ValueError(
            f"Serie troppo corta per timeframe '{timeframe}': "
            f"{len(prices)} candele disponibili, servono almeno {min_required} "
            f"(finestra Hurst={window_c} + buffer 50). "
            f"Fornisci almeno {format_duration(min_required * h_per_candle)} di dati."
        )

    # --- Rolling Hurst ---
    print(
        f"[{timeframe}] Rolling Hurst: "
        f"window={window_c}c ({format_duration(window_c * h_per_candle)}), "
        f"stride={stride_c}c ({format_duration(stride_c * h_per_candle)}), "
        f"min_window={min_win_c}c"
    )
    H_series = rolling_hurst(
        prices,
        window_candles=window_c,
        stride_candles=stride_c,
        min_window_candles=min_win_c,
    )

    # --- Etichetta ticker ---
    ticker_info = label_ticker(H_series)

    # --- Half-life OU ---
    halflife_info = {}
    if 'MEAN_REVERTING' in ticker_info['label']:
        hl_candles = ou_halflife(prices.values)
        if hl_candles is not None:
            hl_hours = hl_candles * h_per_candle
            halflife_info = {
                'half_life_candles':  round(hl_candles, 1),
                'half_life_hours':    round(hl_hours, 1),
                'half_life_human':    format_duration(hl_hours),
                'suggested_lookback_candles': int(np.ceil(hl_candles * 2)),
                'suggested_lookback_human':   format_duration(hl_hours * 2),
            }
        else:
            halflife_info = {'note': 'OU fit non convergente — kappa >= 0'}

    # --- Variance Ratio profile ---
    vr_profile_candles = variance_ratio_profile(prices.values, lags_candles=vr_lags_c)

    # Ricostruisce con chiavi in ore per output leggibile
    vr_profile_hours = {
        candles_to_hours(lag, timeframe): vr
        for lag, vr in vr_profile_candles.items()
    }

    # Scala dominante (prima scala dove VR diverge significativamente da 1)
    vr_scale = None
    for lag_c, vr_val in sorted(vr_profile_candles.items()):
        if not np.isnan(vr_val):
            lag_h = candles_to_hours(lag_c, timeframe)
            if vr_val < 0.90:
                vr_scale = {
                    'type':    'MEAN_REVERTING',
                    'candles': lag_c,
                    'hours':   lag_h,
                    'human':   format_duration(lag_h),
                    'vr':      round(vr_val, 3),
                }
                break
            if vr_val > 1.10:
                vr_scale = {
                    'type':    'MOMENTUM',
                    'candles': lag_c,
                    'hours':   lag_h,
                    'human':   format_duration(lag_h),
                    'vr':      round(vr_val, 3),
                }
                break

    # --- Output ---
    result = {
        'timeframe':         timeframe,
        'h_per_candle':      h_per_candle,
        'n_candles':         len(prices),
        'period_covered':    format_duration(len(prices) * h_per_candle),
        'ticker_label':      ticker_info,
        'H_series':          H_series,
        'half_life':         halflife_info,
        'vr_profile_hours':  vr_profile_hours,
        'vr_dominant_scale': vr_scale,
        'params_used': {
            'hurst_window_hours':    p['hurst_window_hours'],
            'hurst_window_candles':  window_c,
            'hurst_stride_hours':    p['hurst_stride_hours'],
            'hurst_stride_candles':  stride_c,
            'hurst_min_window_candles': min_win_c,
            'vr_lag_hours':          p['vr_lag_hours'],
            'vr_lag_candles':        vr_lags_c,
        },
    }

    _print_summary(result)
    return result


# ---------------------------------------------------------------------------
# 8. SUMMARY PRINT
# ---------------------------------------------------------------------------

def _print_summary(result: dict):
    """Stampa un sommario leggibile dei risultati."""
    tf  = result['timeframe']
    ti  = result['ticker_label']
    vr  = result['vr_profile_hours']
    hl  = result['half_life']
    vrs = result['vr_dominant_scale']
    p   = result['params_used']

    print("\n" + "="*60)
    print(f"  HURST ANALYSIS  —  tf={tf}  |  {result['n_candles']} candele  "
          f"({result['period_covered']})")
    print("="*60)
    print(f"  Finestra Hurst   : {p['hurst_window_candles']}c "
          f"({format_duration(p['hurst_window_hours'])})")
    print(f"  Stride Hurst     : {p['hurst_stride_candles']}c "
          f"({format_duration(p['hurst_stride_hours'])})")
    print("-"*60)
    print(f"  Etichetta asset  : {ti['label']}")
    print(f"  H mediana        : {ti['H_median']}  (±{ti['H_std']})")
    print(f"  H corrente       : {ti['H_current']}")
    print(f"  IC 68%           : {ti['H_ci']}")
    print(f"  Regime drift     : {'⚠️  SÌ' if ti['regime_drift'] else 'NO'}")
    print("-"*60)

    if hl:
        if 'half_life_candles' in hl:
            print(f"  Half-life OU     : {hl['half_life_candles']}c  "
                  f"= {hl['half_life_human']}")
            print(f"  Lookback suggerita: {hl['suggested_lookback_candles']}c  "
                  f"= {hl['suggested_lookback_human']}")
        else:
            print(f"  Half-life OU     : {hl.get('note', 'N/A')}")

    print("-"*60)
    print("  Variance Ratio profile:")
    for lag_h, vr_val in sorted(vr.items()):
        if np.isnan(vr_val):
            continue
        bar = '█' * int(abs(vr_val - 1.0) * 20)
        tag = '← MR' if vr_val < 0.95 else ('→ MOM' if vr_val > 1.05 else '≈ RW')
        print(f"    {format_duration(lag_h):>6s} : VR = {vr_val:.3f}  {bar} {tag}")

    if vrs:
        print(f"\n  Scala dominante  : {vrs['type']} a {vrs['human']}  "
              f"(VR={vrs['vr']})")
    print("="*60 + "\n")


# ---------------------------------------------------------------------------
# ESEMPIO D'USO
# ---------------------------------------------------------------------------

if __name__ == '__main__':
    """
    Test su dati sintetici per tre timeframe diversi.
    La stessa serie OU viene campionata a diversa risoluzione
    per verificare che i risultati siano temporalmente coerenti.

    In produzione sostituisci con la query sul DB:

        from core.models import CandleKPI
        rows = (CandleKPI
                .select()
                .where(
                    CandleKPI.symbol    == 'ADAUSDC',
                    CandleKPI.timeframe == '1h'
                )
                .order_by(CandleKPI.open_time)
                .dicts())
        df = pd.DataFrame(list(rows))
        result = analyze_ticker(df, timeframe='1h')
    """
    np.random.seed(42)

    # Genera serie OU ad alta risoluzione (1h base)
    N_1h       = 3000
    t_1h       = pd.date_range('2025-01-01', periods=N_1h, freq='1h')
    mu, theta, sigma = 0.25, 0.03, 0.0015
    prices_1h  = [mu]
    for _ in range(N_1h - 1):
        dp = theta * (mu - prices_1h[-1]) + sigma * np.random.randn()
        prices_1h.append(prices_1h[-1] + dp)

    # DataFrame 1h
    df_1h = pd.DataFrame({
        'open_time': t_1h,
        'open':   prices_1h,
        'high':   [p * 1.001 for p in prices_1h],
        'low':    [p * 0.999 for p in prices_1h],
        'close':  prices_1h,
        'volume': np.random.randint(1000, 5000, N_1h),
    })

    # DataFrame 4h — ricampionato dalla serie 1h
    df_4h = df_1h.set_index('open_time').resample('4h').agg({
        'open':   'first',
        'high':   'max',
        'low':    'min',
        'close':  'last',
        'volume': 'sum',
    }).dropna().reset_index()

    # Analisi su entrambi i timeframe
    print("\n" + "#"*60)
    print("  TEST 1h")
    print("#"*60)
    r1h = analyze_ticker(df_1h, timeframe='1h')

    print("\n" + "#"*60)
    print("  TEST 4h")
    print("#"*60)
    r4h = analyze_ticker(df_4h, timeframe='4h')

    # Confronto etichette
    print("\n" + "="*60)
    print("  CONFRONTO FINALE")
    print("="*60)
    for tf, res in [('1h', r1h), ('4h', r4h)]:
        ti = res['ticker_label']
        hl = res['half_life']
        hl_str = hl.get('half_life_human', hl.get('note', 'N/A'))
        print(f"  {tf:>3s}  →  {ti['label']:25s}  H={ti['H_median']}  hl={hl_str}")
    print("="*60)


############################################################
  TEST 1h
############################################################
[1h] Rolling Hurst: window=504c (3.0w), stride=24c (1.0d), min_window=50c

  HURST ANALYSIS  —  tf=1h  |  3000 candele  (17.9w)
  Finestra Hurst   : 504c (3.0w)
  Stride Hurst     : 24c (1.0d)
------------------------------------------------------------
  Etichetta asset  : MEAN_REVERTING_STRONG
  H mediana        : 0.0143  (±0.004)
  H corrente       : 0.0252
  IC 68%           : (0.0103, 0.0182)
  Regime drift     : ⚠️  SÌ
------------------------------------------------------------
  Half-life OU     : 21.6c  = 21.6h
  Lookback suggerita: 44c  = 1.8d
------------------------------------------------------------
  Variance Ratio profile:
      4.0h : VR = 0.938  █ ← MR
      8.0h : VR = 0.878  ██ ← MR
     16.0h : VR = 0.793  ████ ← MR
      1.0d : VR = 0.715  █████ ← MR
      2.0d : VR = 0.566  ████████ ← MR
      4.0d : VR = 0.318  █████████████ ← MR
 